In [ ]:
#check we ar eon right node
import socket
print(f"Notebook is running on: {socket.gethostname()}")

In [ ]:
import numpy as np 
import xarray as xr
import pandas as pd
import geopandas as gpd
import regionmask
import glob
import os
import matplotlib.pyplot as plt

In [ ]:
# 1. Setup BA Shapes and Mask
shapefile_path = "control-areas-shapefile copy/Control_Areas.shp"
ba_shapes = gpd.read_file(shapefile_path)

ba_shapes = ba_shapes.to_crs("EPSG:4326")
ba_shapes['ID'] = pd.to_numeric(ba_shapes['ID'], errors='coerce')
ba_shapes = ba_shapes.dropna(subset=['ID'])
ba_shapes['ID'] = ba_shapes['ID'].astype(int)


# Check the range of  Shapefile
bbox = ba_shapes.total_bounds # Returns [min_lon, min_lat, max_lon, max_lat]
print(f"Shapefile Lat Range: {bbox[1]} to {bbox[3]}")
print(f"Shapefile Lon Range: {bbox[0]} to {bbox[2]}")

# Total BAs in your shapefile
total_bas = len(ba_shapes)
print(f"Total BAs in Shapefile: {total_bas}")

# List the unique names of the BAs you are currently looking at
print("Sample BAs in your dataset:")
print(ba_shapes['NAME'].head().tolist())


# Basic plot to see the geometry
ba_shapes.plot(figsize=(10, 10), edgecolor='black', color='lightblue')
plt.title("Spatial Distribution of Control Areas")
plt.show()

In [ ]:
import os

# Start from a known high-level directory and check what's inside
base_path = "/glade/campaign/collections/rda/data/d559000/wy2016/"

if os.path.exists(base_path):
    print(f"Contents of {base_path}:")
    print(os.listdir(base_path))
else:
    print("Base path does not exist. Check your permissions or the collection ID (d559000).")

In [ ]:
#load 2018 files
# 1. Get Jan 2018 through Sept 2018 from wy2018
#jan_to_sep = glob.glob("/glade/campaign/collections/rda/data/d559000/wy2018/2018/201801*/wrf3d*.nc")
#oct_to_dec = glob.glob("/glade/campaign/collections/rda/data/d559000/wy2018/2017*/wrf3d*.nc")
#all_2018_files = sorted(jan_to_sep + oct_to_dec)

##01/2018 test run
import glob
import xarray as xr
import matplotlib.pyplot as plt

# 1. Corrected Path
path_to_search = "/glade/campaign/collections/rda/data/d559000/wy2016/201601*/wrf3d*.nc"
jan_files = sorted(glob.glob(path_to_search))

if not jan_files:
    print("Still no files found. Let's check if the .nc files are directly in 201601:")
    # Alternative check:
    path_alt = "/glade/campaign/collections/rda/data/d559000/wy2016/201601/wrf3d*.nc"
    jan_files = sorted(glob.glob(path_alt))

if jan_files:
    print("Success! Found {len(jan_files)} files for January.")
    all_2016_files = jan_files
    
    # 2. Open and Inspect
    ds = xr.open_dataset(all_2016_files[0])
    
    # Select surface temperature (TK) at the first time step
    tk_surface = ds['TK'].isel(Time=0, bottom_top=0)
    
    # 3. Plot
    plt.figure(figsize=(10, 6))
    tk_surface.plot()
    plt.title("Surface Temperature (TK) - Jan 2016")
    plt.show()
else:
    print(" No files found. Double check the filename prefix (is it 'wrf3d' or something else?)")

#end delete later
print(f"Filtered file count for 2016: {len(all_2016_files)}") # Should be ~8,784
print(f"First file: {all_2016_files[0]}") 
print(f"Last file: {all_2016_files[-1]}")

import xarray as xr

# Print the structure
print(ds)

# Specifically look at the spatial coordinates
print(ds[['XLAT', 'XLONG']])

import xarray as xr

# Open one of your 2016 files
ds = xr.open_dataset(all_2016_files[0])

# TK usually has dimensions: (Time, bottom_top, south_north, west_east)
# We select the first vertical level (z=0) and the first time step
tk_surface = ds['TK'].isel(Time=0, bottom_top=0)

print(tk_surface)

tk_surface.plot(figsize=(10, 6))
plt.title("Surface Temperature (TK) from WRF 2016 Sample File")
plt.xlabel("Longitude Index")
plt.ylabel("Latitude Index")
plt.show()

In [ ]:
#verify spatial alignment of sha[efile and WFR
# 1. Prepare the figure
fig, ax = plt.subplots(figsize=(15, 10))

# 2. Plot with shading='nearest'
# We also use .squeeze() to safely handle the Time dimension if it exists
lons = ds.XLONG.squeeze()
lats = ds.XLAT.squeeze()
data = tk_surface.squeeze()

cbar = ax.pcolormesh(lons, lats, data, cmap='magma', shading='nearest')
plt.colorbar(cbar, ax=ax, label='Temperature (K)')

# 3. Overlay the Shapefile Boundaries
# Note: Ensure ba_shapes is in EPSG:4326 to match WRF's lat/lon
ba_shapes.boundary.plot(ax=ax, color='cyan', linewidth=0.5, alpha=0.8)

plt.title("Spatial Alignment Check: WRF vs. Balancing Authorities")
plt.show()

Filter out BA for Alaska/Hawaii since CONSUS does not cover

In [ ]:
import os
import glob
import numpy as np
import xarray as xr
import geopandas as gpd
import regionmask

# hdf5 locking workarounf for casper
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

#loda shapefiles
shapefile_path = "control-areas-shapefile copy/Control_Areas.shp"
ba_shapes = gpd.read_file(shapefile_path)
ba_shapes = ba_shapes.to_crs("EPSG:4326")
ba_shapes['ID'] = pd.to_numeric(ba_shapes['ID'], errors='coerce')
ba_shapes = ba_shapes.dropna(subset=['ID'])
ba_shapes['ID'] = ba_shapes['ID'].astype(int)

# List the unique names of the BAs you are currently looking at
print("Sample BAs in your dataset:")
print(ba_shapes['NAME'].head().tolist())
unique_bas = len(ba_shapes['NAME'].unique())
print(f"Unique BAs in Shapefile: {unique_bas}")

# regionmask for merging 
ba_regions = regionmask.from_geopandas(
    ba_shapes,
    numbers="ID",
    names="NAME",
    overlap=False
)

# Base directory based on successful check
base_path = "/glade/campaign/collections/rda/data/d559000/wy2016/"

# 1. Get Oct-Dec 2015
# This looks for folders starting with 201510, 201511, 201512
oct_to_dec_2015 = glob.glob(f"{base_path}20151[012]*/wrf3d*.nc")

# 2. Get Jan-Sep 2016
# This looks for folders starting with 201601 through 201609
jan_to_sep_2016 = glob.glob(f"{base_path}20160[1-9]*/wrf3d*.nc")

# Combine and sort
all_2016_files = sorted(oct_to_dec_2015 + jan_to_sep_2016)

# Verification
print(f"Total files found: {len(all_2016_files)}")
if all_2016_files:
    print(f"First file: {all_2016_files[0]}")
    print(f"Last file:  {all_2016_files[-1]}")

# Test for corrupted/bad files
bad_files = []
for f in all_2016_files:
    try:
        xr.open_dataset(f)
    except Exception as e:
        print(f"Bad file: {f}")
        bad_files.append(f)

# Filter out bad files
all_2016_files = [f for f in all_2016_files if f not in bad_files]
print(f"Filtered file count: {len(all_2016_files)}")

#load data in batches
# Use engine 'h5netcdf' for stability on Casper/Glade
ds = xr.open_mfdataset(
    all_2016_files[:],   # can add all files/few for testing
    combine='by_coords',
    engine='h5netcdf',
    parallel=False,
    chunks={'Time': 24}
)

# BA mask
mask = ba_regions.mask(ds.XLONG, ds.XLAT)

# compute spatial average per BA

# If your spatial dims were automatically stacked
spatial_avg = (
    ds['TK']
    .isel(bottom_top=0)  # surface level
    .stack(space=('south_north', 'west_east'))  # stack into one dim
    .groupby(mask.stack(space=('south_north', 'west_east')))  # same stack for mask
    .mean(dim='space')  # reduce over stacked spatial dim
)

#7. convert and save
# convert to df
final_df_2016 = spatial_avg.to_dataframe().reset_index()

# Rename temperature column
final_df_2016 = final_df_2016.rename(columns={spatial_avg.name: 'avg_temp_K'})

# Merge BA names from regionmask
ba_lookup = pd.DataFrame({
    'ID': ba_regions.numbers,   # numeric IDs
    'BA_NAME': ba_regions.names # corresponding names
})

# Use 'mask' column, not 'group'
final_df_2016 = final_df_2016.merge(ba_lookup, left_on='mask', right_on='ID', how='left')

# Drop the temporary 'mask' column (we already have ID and BA_NAME)
final_df_2016 = final_df_2016.drop(columns=['mask'])

# Optional: reorder columns
final_df_2016 = final_df_2016[['Time', 'ID', 'BA_NAME', 'avg_temp_K']]

print(final_df_2016.head())

#save 2016 file
output_csv = os.path.join(os.path.expanduser("~"), "BA_Load_Temp_Merge_2016.csv")
final_df_2016.to_csv(output_csv, index=False)
